# 🛡️ Face Anti-Spoofing — 18: 웹캠 공격 데이터 수집 & Fine-tuning v3
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---

## 🎯 목표
CelebA-Spoof real 이미지를 **프린트/모니터**에 출력 후 웹캠으로 촬영하여  
웹캠 도메인의 Print/Replay 공격 이미지를 직접 수집 → Fine-tuning

## 📋 전체 흐름
```
CelebA real 50장 추출
    ↓
[직접 수행] 프린트 출력 → 웹캠 촬영 → webcam_print/ 저장
[직접 수행] 모니터 띄우기 → 웹캠 촬영 → webcam_replay/ 저장
    ↓
Fine-tuning v3 (webcam_print + webcam_replay 포함)
    ↓
검증 & 모델 저장
```

## ✅ 체크리스트
- [ ] Cell 1: Drive 마운트 + 경로 설정
- [ ] Cell 2: 출력용 이미지 50장 추출
- [ ] Cell 3: [수동] 웹캠 촬영 앱 실행
- [ ] Cell 4: 수집 데이터 확인
- [ ] Cell 5: Fine-tuning v3 데이터 구성
- [ ] Cell 6: Phase A — Head 재학습
- [ ] Cell 7: 검증 & 모델 저장

## Cell 1 — Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

BASE          = '/content/drive/MyDrive/face-anti-spoofing'
CROP_DIR      = f'{BASE}/data/cropped'
WEBCAM_DIR    = f'{BASE}/data/webcam_live'      # 기존 웹캠 Live
WC_PRINT_DIR  = f'{BASE}/data/webcam_print'     # 웹캠 Print 공격 (수집 예정)
WC_REPLAY_DIR = f'{BASE}/data/webcam_replay'    # 웹캠 Replay 공격 (수집 예정)
PRINT_TARGET  = f'{BASE}/data/print_targets'    # 출력용 이미지
MODEL_DIR     = f'{BASE}/models'
REPORT_DIR    = f'{BASE}/reports'

for d in [WC_PRINT_DIR, WC_REPLAY_DIR, PRINT_TARGET]:
    os.makedirs(d, exist_ok=True)

print('\n=== 현재 데이터 현황 ===')
dirs = {
    'webcam_live'   : WEBCAM_DIR,
    'webcam_print'  : WC_PRINT_DIR,
    'webcam_replay' : WC_REPLAY_DIR,
    'CelebA live'   : f'{CROP_DIR}/live',
    'CelebA print'  : f'{CROP_DIR}/print',
    'CelebA replay' : f'{CROP_DIR}/replay',
    'CelebA mask'   : f'{CROP_DIR}/mask',
}
for name, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith(('.jpg','.png'))]) if os.path.exists(d) else 0
    status = '✅' if n > 0 else '⬜ 비어있음'
    print(f'  {name:<16}: {n:>4}장  {status}')

## Cell 2 — 출력용 이미지 50장 추출

> CelebA-Spoof real 이미지 중 **선명한 순으로 50장** 추출  
> → Drive `data/print_targets/` 에 512×512로 저장  
> → 다운로드 후 **프린트 출력** 또는 **모니터/핸드폰에 띄우기**

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path
import matplotlib.pyplot as plt

random.seed(42)

LIVE_DIR = f'{CROP_DIR}/live'
imgs     = list(Path(LIVE_DIR).glob('*.jpg'))
random.shuffle(imgs)

# Laplacian 높은 순 (선명한 이미지) 50장 선택
scored = []
for p in imgs[:300]:
    img  = cv2.imread(str(p))
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lap  = cv2.Laplacian(gray, cv2.CV_64F).var()
    scored.append((str(p), lap))

scored.sort(key=lambda x: x[1], reverse=True)
selected = scored[:50]

# 512×512로 저장 (프린트/모니터용)
saved_paths = []
for i, (p, lap) in enumerate(selected):
    img      = cv2.imread(p)
    img_large = cv2.resize(img, (512, 512))
    out_path  = f'{PRINT_TARGET}/target_{i:03d}.jpg'
    cv2.imwrite(out_path, img_large)
    saved_paths.append(out_path)

print(f'✅ 출력용 이미지 {len(selected)}장 저장: {PRINT_TARGET}')

# 샘플 미리보기
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('출력용 이미지 샘플 (상위 10장)', fontsize=13)
for ax, (p, lap) in zip(axes.flat, selected[:10]):
    img = cv2.imread(p)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f'Lap={lap:.0f}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\n=== 다음 단계 ===')
print('1. Drive → data/print_targets/ 폴더 열기')
print('2. 이미지 다운로드')
print('3. [Print 수집] 프린트 출력 → 웹캠 앞에 들이대며 촬영')
print('4. [Replay 수집] 모니터/핸드폰에 띄우기 → 웹캠으로 촬영')
print('5. 촬영 이미지를 각각 webcam_print/, webcam_replay/ 에 저장')
print('6. Cell 3 실행 (웹캠 촬영 앱)')

## Cell 3 — [수동] 웹캠 촬영 앱 실행

> **촬영 방법:**
> 1. 아래 셀 실행 → ngrok URL 열기
> 2. Print 수집: 프린트된 사진을 웹캠 앞에 → 스페이스바로 캡처 30~50장
> 3. Replay 수집: 모니터/핸드폰 화면을 웹캠 앞에 → 스페이스바로 캡처 30~50장
> 4. 촬영된 이미지를 Drive의 `webcam_print/`, `webcam_replay/` 폴더로 이동

In [ ]:
import subprocess, time, sys, os
from pyngrok import ngrok, conf

NGROK_TOKEN = ''  # ← ngrok authtoken 입력

# 웹캠 촬영 앱 생성
APP_CODE = '''
import streamlit as st
import cv2
import numpy as np
from pathlib import Path
import os

BASE = "/content/drive/MyDrive/face-anti-spoofing"

st.title("📸 웹캠 공격 데이터 수집")

mode = st.radio("수집 모드", ["Print Attack", "Replay Attack"])
save_dir = f"{BASE}/data/webcam_print" if mode == "Print Attack" else f"{BASE}/data/webcam_replay"
os.makedirs(save_dir, exist_ok=True)

n_saved = len(list(Path(save_dir).glob("*.jpg")))
st.info(f"현재 저장된 이미지: {n_saved}장 → 목표: 50장")
st.progress(min(n_saved / 50, 1.0))

img_file = st.camera_input("📷 촬영 (화면이 꽉 차도록)")

if img_file:
    import time
    bytes_data = img_file.getvalue()
    arr  = np.frombuffer(bytes_data, np.uint8)
    img  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img  = cv2.resize(img, (224, 224))
    fname = f"{save_dir}/wcm_{mode.split()[0].lower()}_{int(time.time()*1000)}.jpg"
    cv2.imwrite(fname, img)
    st.success(f"✅ 저장: {os.path.basename(fname)} (총 {n_saved+1}장)")
    st.image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), width=224)
'''

with open('/content/collect_app.py', 'w') as f:
    f.write(APP_CODE)

if not NGROK_TOKEN:
    print('⚠️ NGROK_TOKEN 입력 필요')
else:
    conf.get_default().auth_token = NGROK_TOKEN
    !pip install -q streamlit pyngrok
    os.system('pkill -f streamlit 2>/dev/null')
    time.sleep(1)

    proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', '/content/collect_app.py',
         '--server.port', '8502', '--server.headless', 'true'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(4)
    tunnel = ngrok.connect(8502)
    print('=' * 50)
    print(f'📸 수집 앱 URL: {tunnel.public_url}')
    print('=' * 50)
    print('1. Print Attack 모드: 프린트 사진 웹캠 앞에 대고 촬영')
    print('2. Replay Attack 모드: 모니터/핸드폰 화면 웹캠 앞에 대고 촬영')
    print('각 50장씩 목표!')

## Cell 4 — 수집 데이터 확인 & Laplacian/FFT 분포

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def measure_stats(img_bgr):
    gray  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap   = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    gray_f = gray.astype(np.float32)
    f      = np.fft.fftshift(np.fft.fft2(gray_f))
    mag    = np.abs(f)
    h, w   = mag.shape
    cy, cx = h//2, w//2
    r      = min(h,w)//6
    Y, X   = np.ogrid[:h, :w]
    d2     = (Y-cy)**2 + (X-cx)**2
    return lap, float(mag[d2 > r**2].mean())

def get_stats(d, max_n=50):
    paths = list(Path(d).glob('*.jpg'))[:max_n]
    laps, ffts = [], []
    for p in paths:
        raw = np.fromfile(str(p), dtype=np.uint8)
        img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
        if img is None: continue
        l, f = measure_stats(img)
        laps.append(l); ffts.append(f)
    return np.array(laps), np.array(ffts)

print('=== 수집 현황 ===')
for name, d in [('webcam_print', WC_PRINT_DIR), ('webcam_replay', WC_REPLAY_DIR)]:
    n = len(list(Path(d).glob('*.jpg')))
    status = '✅ 충분' if n >= 30 else f'⚠️ {50-n}장 더 필요'
    print(f'  {name}: {n}장  {status}')

# 분포 비교
categories = {
    'webcam_live'   : WEBCAM_DIR,
    'webcam_print'  : WC_PRINT_DIR,
    'webcam_replay' : WC_REPLAY_DIR,
    'CelebA_print'  : f'{CROP_DIR}/print',
    'CelebA_replay' : f'{CROP_DIR}/replay',
}

print('\n=== Laplacian / FFT 분포 ===')
print(f'  {"카테고리":<16} {"Lap 평균":>10} {"FFT 평균":>10} {"장수":>6}')
print('  ' + '-'*46)
for name, d in categories.items():
    if not os.path.exists(d): continue
    laps, ffts = get_stats(d)
    if len(laps) == 0: continue
    print(f'  {name:<16} {np.mean(laps):>10.1f} {np.mean(ffts):>10.1f} {len(laps):>6}')

## Cell 5 — Fine-tuning v3 데이터 구성

> **v2 대비 추가:**  
> - `webcam_print/`  → label_bin=1, label_spoof=1  
> - `webcam_replay/` → label_bin=1, label_spoof=2

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

IMG_SIZE = 224

def load_img(path):
    raw = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img is None: return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0

def load_from_dir(d, max_n=None, label_bin=0, label_spoof=0):
    paths = sorted(Path(d).glob('*.jpg')) + sorted(Path(d).glob('*.png'))
    if max_n:
        paths = random.sample(paths, min(max_n, len(paths)))
    imgs, bins, spoofs = [], [], []
    for p in paths:
        img = load_img(p)
        if img is not None:
            imgs.append(img)
            bins.append(label_bin)
            spoofs.append(label_spoof)
    print(f'  로드: {len(imgs)}장 ← {os.path.basename(d)}')
    return imgs, bins, spoofs

print('=== 데이터 로드 ===')

# ── Live ──────────────────────────────────────────────────
wcm_imgs, wcm_b, wcm_s = load_from_dir(WEBCAM_DIR,   label_bin=0, label_spoof=0)
n_webcam = len(wcm_imgs)
cel_imgs, cel_b, cel_s = load_from_dir(f'{CROP_DIR}/live', max_n=n_webcam, label_bin=0, label_spoof=0)

# ── 웹캠 공격 (신규) ──────────────────────────────────────
wcp_imgs, wcp_b, wcp_s = load_from_dir(WC_PRINT_DIR,  label_bin=1, label_spoof=1)
wcr_imgs, wcr_b, wcr_s = load_from_dir(WC_REPLAY_DIR, label_bin=1, label_spoof=2)

# ── CelebA 공격 ───────────────────────────────────────────
prt_imgs, prt_b, prt_s = load_from_dir(f'{CROP_DIR}/print',  max_n=300, label_bin=1, label_spoof=1)
rpl_imgs, rpl_b, rpl_s = load_from_dir(f'{CROP_DIR}/replay', max_n=300, label_bin=1, label_spoof=2)
msk_imgs, msk_b, msk_s = load_from_dir(f'{CROP_DIR}/mask',   max_n=300, label_bin=1, label_spoof=3)

# ── 합치기 ────────────────────────────────────────────────
all_imgs   = wcm_imgs + cel_imgs + wcp_imgs + wcr_imgs + prt_imgs + rpl_imgs + msk_imgs
all_binary = wcm_b   + cel_b    + wcp_b    + wcr_b    + prt_b    + rpl_b    + msk_b
all_spoof  = wcm_s   + cel_s    + wcp_s    + wcr_s    + prt_s    + rpl_s    + msk_s

X     = np.array(all_imgs,   dtype=np.float32)
y_bin = np.array(all_binary, dtype=np.float32)
y_sp  = np.array(all_spoof,  dtype=np.int32)

print(f'\n=== 전체 데이터 ===')
print(f'  총: {len(X)}장')
print(f'  Live(0): {(y_bin==0).sum()}장')
print(f'  Fake(1): {(y_bin==1).sum()}장')
print(f'    Print:  {(y_sp==1).sum()}장 (CelebA+웹캠)')
print(f'    Replay: {(y_sp==2).sum()}장 (CelebA+웹캠)')
print(f'    Mask:   {(y_sp==3).sum()}장')

# 오버샘플링 1:1.4
idx = list(range(len(X)))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.3, random_state=42, stratify=y_bin)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.5, random_state=42, stratify=y_bin[tmp_idx])

X_tr, y_bin_tr, y_sp_tr = X[tr_idx], y_bin[tr_idx], y_sp[tr_idx]
X_va, y_bin_va, y_sp_va = X[va_idx], y_bin[va_idx], y_sp[va_idx]
X_te, y_bin_te, y_sp_te = X[te_idx], y_bin[te_idx], y_sp[te_idx]

# 오버샘플링 1:1.4
live_idx = np.where(y_bin_tr == 0)[0]
fake_idx = np.where(y_bin_tr == 1)[0]
target_live  = int(len(fake_idx) / 1.4)
live_idx_over = np.random.choice(live_idx, size=target_live, replace=True)
balanced_idx  = np.concatenate([live_idx_over, fake_idx])
np.random.shuffle(balanced_idx)

X_tr_b     = X_tr[balanced_idx]
y_bin_tr_b = y_bin_tr[balanced_idx]
y_sp_tr_b  = y_sp_tr[balanced_idx]

print(f'\n=== 오버샘플링 후 Train ===')
print(f'  총: {len(X_tr_b)}장  Live:{(y_bin_tr_b==0).sum()} Fake:{(y_bin_tr_b==1).sum()}')
print(f'  Val: {len(X_va)}장  Test: {len(X_te)}장')

## Cell 6 — Fine-tuning v3 (Phase A: Head 재학습)

In [ ]:
import tensorflow as tf

# stage2_best.h5 (원본)에서 시작
MODEL_PATH = f'{MODEL_DIR}/stage2_best.h5'
model = tf.keras.models.load_model(MODEL_PATH)
print('✅ 모델 로드:', MODEL_PATH)

# Head만 해동
model.trainable = True
for layer in model.layers:
    layer.trainable = False
for layer in model.layers:
    if layer.name in ['binary', 'spoof', 'dense', 'dense_1',
                      'global_average_pooling2d', 'dropout',
                      'dropout_1', 'dropout_2', 'shared',
                      'batch_normalization']:
        layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f'학습 가능 레이어: {trainable}/{len(model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'binary': 'binary_crossentropy', 'spoof': 'sparse_categorical_crossentropy'},
    loss_weights={'binary': 0.7, 'spoof': 0.3},
    metrics={'binary': ['accuracy'], 'spoof': ['accuracy']}
)

cb = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3,
        restore_best_weights=True, mode='min'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/stage2_webcam_v3.h5',
        monitor='val_loss', save_best_only=True,
        verbose=1, mode='min'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-5, verbose=1, mode='min'
    )
]

print('\n=== Fine-tuning v3 시작 ===')
hist = model.fit(
    X_tr_b, {'binary': y_bin_tr_b, 'spoof': y_sp_tr_b},
    validation_data=(X_va, {'binary': y_bin_va, 'spoof': y_sp_va}),
    epochs=15,
    batch_size=32,
    callbacks=cb,
    verbose=1
)

print('\n✅ 학습 완료')
print(f'  최고 val_binary_accuracy: {max(hist.history["val_binary_accuracy"]):.4f}')
print(f'  최고 val_spoof_accuracy:  {max(hist.history["val_spoof_accuracy"]):.4f}')

## Cell 7 — 검증 & threshold 탐색 & 모델 저장

In [ ]:
import numpy as np

# threshold 탐색
preds_te = model.predict(X_te, verbose=0)
probs_te = preds_te[0][:,0] if isinstance(preds_te, list) else preds_te[:,0]

print(f'{"threshold":>10} {"Live→REAL":>10} {"Fake→FAKE":>10}')
print('-' * 34)
best_th, best_score = 0.5, 0
for th in np.arange(0.30, 0.80, 0.05):
    v = (probs_te >= th).astype(int)
    live_real = (v[y_bin_te==0]==0).mean()
    fake_fake = (v[y_bin_te==1]==1).mean()
    score = min(live_real, fake_fake)
    marker = ' ← 최적' if score > best_score else ''
    if score > best_score:
        best_score = score
        best_th = float(th)
    print(f'{th:>10.2f} {live_real:>10.1%} {fake_fake:>10.1%}{marker}')

# 카테고리별 검증
print(f'\n=== threshold={best_th:.2f} 카테고리별 검증 ===')
categories = {
    'webcam_live'   : (WEBCAM_DIR,             '≥95%', True),
    'webcam_print'  : (WC_PRINT_DIR,           '≥90%', False),
    'webcam_replay' : (WC_REPLAY_DIR,          '≥90%', False),
    'CelebA_live'   : (f'{CROP_DIR}/live',     '≥85%', True),
    'CelebA_print'  : (f'{CROP_DIR}/print',    '≥95%', False),
    'CelebA_replay' : (f'{CROP_DIR}/replay',   '≥80%', False),
    'CelebA_mask'   : (f'{CROP_DIR}/mask',     '≥90%', False),
}

print(f'  {"카테고리":<16} {"REAL%":>7} {"FAKE%":>7} {"기준":>6} {"판정":>6}')
print('  ' + '-'*46)
all_pass = True
for cat, (d, crit, is_live) in categories.items():
    if not os.path.exists(d) or len(os.listdir(d)) == 0:
        print(f'  {cat:<16} → 스킵 (데이터 없음)')
        continue
    imgs, _, _ = load_from_dir(d, max_n=50, label_bin=0, label_spoof=0)
    X_cat = np.array(imgs)
    preds = model.predict(X_cat, verbose=0)
    probs = preds[0][:,0] if isinstance(preds, list) else preds[:,0]
    v     = (probs >= best_th).astype(int)
    real_pct = (v==0).mean()
    fake_pct = (v==1).mean()
    th_val   = int(crit[1:-1]) / 100
    actual   = real_pct if is_live else fake_pct
    passed   = actual >= th_val
    if not passed: all_pass = False
    status = '✅' if passed else '❌'
    print(f'  {cat:<16} {real_pct:>7.1%} {fake_pct:>7.1%} {crit:>6} {status}')

print(f'\n{"🎉 전체 합격!" if all_pass else "⚠️ 일부 미달"}')
print(f'\n=== 최종 모델 저장 ===')
model.save(f'{MODEL_DIR}/stage2_webcam_v3.h5')
print(f'✅ stage2_webcam_v3.h5 저장 완료')
print(f'\n→ xai_explainer.py 수정 필요:')
print(f'  MODEL_PATH: stage2_webcam_v2.h5 → stage2_webcam_v3.h5')
print(f'  threshold:  0.65 → {best_th:.2f}')